# Multi-Agent MeSH-Guided Concept Retrieval RAG

Runs Architecture 4: MeSH-Guided Concept Retrieval — a 3-agent sequential pipeline
that extracts medical concepts from each question using an LLM, pre-filters the chunk
corpus to only documents whose MeSH terms overlap with those concepts, then runs
cosine similarity retrieval within the filtered sub-corpus.

**Pipeline:** Question → Agent 1 (Medical Concept Extractor) → Agent 2 (MeSH-Filtered Cosine Retriever) → Agent 3 (Answer Generator) → Answer  
**Evaluation:** RAGAS and DeepEval metrics

In [27]:
import sys
sys.path.append("..")

import os
import re
import time
import json
import numpy as np
import pandas as pd
from datetime import datetime
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.documents import Document
import config
from ast import literal_eval
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
from ragas import SingleTurnSample
from deepeval.evaluate import DisplayConfig, AsyncConfig
from deepeval.test_case import LLMTestCase
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import (
    FaithfulnessMetric, ContextualRecallMetric,
    ContextualPrecisionMetric, AnswerRelevancyMetric,
)
import deepeval
import instructor
from groq import AsyncGroq
from ragas.llms.base import InstructorLLM
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()
pd.set_option('display.html.use_mathjax', False)

import logging
logging.basicConfig(level=logging.ERROR)

/tmp/ipykernel_6669/3053808417.py:20: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams
/tmp/ipykernel_6669/3053808417.py:21: DeprecationWarning: Importing NonLLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMContextRecall
  from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
/tmp/ipykernel_6669/3053808417.py:21: DeprecationWarning: Importing NonLLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMContextPrecisionWithReference
  from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference, BleuScore, RougeScore
/t

## Load Vector Store

Loaders for each vector DB ingested by `ingestion_pipeline.ipynb`. All functions are
read-only — they never re-embed or re-write.

In [8]:
def load_chroma(embeddings, db_name=None, persist_dir=None):
    """Load an existing ChromaDB collection from disk.

    Args:
        embeddings: LangChain embeddings instance (must match what was used during ingestion).
        db_name: Collection name; defaults to {DEFAULT_EMBEDDING}_pubmed_chroma.
        persist_dir: Override storage path (defaults to vectorstores/{db_name}).

    Returns:
        Chroma vector store instance.
    """
    from langchain_chroma import Chroma

    db_name = db_name or f"{config.DEFAULT_EMBEDDING}_pubmed_chromadb"
    persist_dir = persist_dir or str(config.VECTORSTORE_DIR / db_name)
    print(f"Loading ChromaDB from {persist_dir}")
    return Chroma(
        collection_name=db_name,
        persist_directory=persist_dir,
        embedding_function=embeddings,
    )

In [9]:
def get_cosine_retriever(vector_store, k=None):
    """Method to build a cosine similarity based retriever for given vector store
    Args:
        vector_store: Langchain vector store object which has method as_retriever
        k: Top k items to be retrieved
    Returns:
        retriever object
    """
    k = k or config.TOP_K
    return vector_store.as_retriever(search_kwargs={"k": k})

In [10]:
def get_groq_llm(model=None, api_key=None):
    return ChatGroq(
        model=model or config.LLM_MODEL,
        api_key=api_key or config.GROQ_API_KEY,
    )


class GroqKeyRotator:
    """Cycles through config.GROQ_API_KEYS, rebuilding the LLM client on each rotation."""

    def __init__(self, model=None):
        if not config.GROQ_API_KEYS:
            raise ValueError("config.GROQ_API_KEYS is empty — set GROQ_API_KEY or GROQ_API_KEYS in .env")
        self.api_keys = config.GROQ_API_KEYS
        self.model = model or config.LLM_MODEL
        self.current_idx = 0
        print(f"Initialized GroqKeyRotator with {len(self.api_keys)} API key(s)")

    def get_llm(self):
        """Return a ChatGroq instance using the current API key."""
        return ChatGroq(
            model=self.model,
            api_key=self.api_keys[self.current_idx],
        )

    def rotate(self):
        """Advance to the next key, wrapping around."""
        self.current_idx = (self.current_idx + 1) % len(self.api_keys)
        print(f"Rotated to API key index {self.current_idx}")


def is_rate_limit_error(e):
    """Check if rate limit error is reached by checking the error message."""
    msg = str(e).lower()
    return any(kw in msg for kw in [
        "rate_limit", "rate limit", "429", "too many requests",
        "tokens per", "token limit", "exceeded",
    ])


RAG_PROMPT_TEMPLATE = """Use the following research contexts to answer the question.

Context:
{context}

Question: {question}

Answer based only on the provided context. Be precise and evidence-based.

Answer:"""

RAG_PROMPT = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"],
)


def build_rag_chain(llm):
    return RAG_PROMPT | llm

## Agent 1: Medical Concept Extractor

Extracts 3–5 clinically significant medical concepts from the question using one LLM call.
The output concept list is used in the next step to pre-filter the chunk corpus by MeSH term overlap.

In [11]:
CONCEPT_EXTRACTION_PROMPT_TEMPLATE = """You are a biomedical entity extractor.
Given a biomedical research question, extract the 3 to 5 most clinically significant medical concepts.
These may include diseases, drugs, procedures, biomarkers, anatomical structures, or patient populations.
Return only a comma-separated list of concept terms. Do not include explanations or punctuation other than commas.

Question: {question}

Concepts:"""

CONCEPT_EXTRACTION_PROMPT = PromptTemplate(
    template=CONCEPT_EXTRACTION_PROMPT_TEMPLATE,
    input_variables=["question"],
)


def build_concept_extraction_chain(llm):
    return CONCEPT_EXTRACTION_PROMPT | llm


def parse_concepts(raw_output):
    """Parse comma-separated concept strings from LLM output into a cleaned list."""
    text = raw_output.content if hasattr(raw_output, "content") else str(raw_output)
    concepts = [c.strip().lower() for c in text.split(",") if c.strip()]
    return concepts

## Agent 2: MeSH-Filtered Cosine Retriever

Pre-loads the full chunk corpus from ChromaDB (documents, embeddings, metadata).
For each question, filters chunks to those whose `mesh_terms` metadata list shares
at least one term with the extracted concept list. Cosine similarity is then computed
within the filtered sub-corpus using numpy, avoiding a second round-trip to the vector store.
If the filtered set contains fewer than `min_filtered` chunks, the filter is released
and retrieval proceeds against the full corpus as a fallback.

In [12]:
def build_mesh_corpus(vector_store, embeddings):
    """Load all chunks from ChromaDB and compute embeddings once in memory for MeSH-filtered retrieval.

    Returns a dict with keys: ids, embeddings, documents, metadatas.
    The embeddings array has shape (n_chunks, embedding_dim).
    """
    print("Loading corpus from ChromaDB (this is a one-time operation)...")
    corpus = vector_store.get(include=["documents", "metadatas"])
    corpus["embeddings"] = np.asarray(
        embeddings.embed_documents(corpus["documents"]),
        dtype=np.float32,
    )
    print(f"Corpus loaded: {len(corpus['ids'])} chunks, docs = {len(corpus['documents'])}")
    return corpus


def _cosine_similarity(query_vec, doc_vecs):
    """Compute cosine similarity between a query vector and a matrix of document vectors."""
    query_norm = query_vec / (np.linalg.norm(query_vec) + 1e-8)
    doc_norms = doc_vecs / (np.linalg.norm(doc_vecs, axis=1, keepdims=True) + 1e-8)
    return doc_norms @ query_norm


def _mesh_overlaps(metadata, concepts):
    """Check if a chunk's mesh_terms metadata overlaps with the extracted concept list.

    Uses case-insensitive substring matching: a chunk passes if any mesh_term
    is a substring of any concept or vice versa.
    """
    raw = metadata.get("mesh_terms", "")
    if not raw:
        return False
    # mesh_terms may be stored as a JSON string or comma-separated list
    try:
        mesh_list = json.loads(raw) if isinstance(raw, str) and raw.startswith("[") else [t.strip() for t in raw.split(",")]
    except Exception:
        mesh_list = [t.strip() for t in str(raw).split(",")]
    mesh_lower = [m.lower() for m in mesh_list if m]
    for mesh in mesh_lower:
        for concept in concepts:
            if mesh in concept or concept in mesh:
                return True
    return False


def mesh_filtered_retrieval(corpus, embeddings_model, question, concepts, k=5, min_filtered=10):
    """Retrieve top-k chunks using MeSH-filtered cosine similarity.

    Filters the corpus to chunks whose MeSH terms overlap with `concepts`,
    then ranks the filtered set by cosine similarity to the question embedding.
    Falls back to the full corpus if the filtered subset is smaller than `min_filtered`.

    Args:
        corpus: Dict returned by build_mesh_corpus with embeddings, documents, metadatas, ids.
        embeddings_model: HuggingFaceEmbeddings instance used to embed the query.
        question: Raw question string.
        concepts: List of lowercase concept strings from the concept extractor.
        k: Number of top chunks to return.
        min_filtered: Minimum filtered corpus size before falling back to full corpus.

    Returns:
        Tuple of (list of LangChain Document objects, int mesh_filtered_count, bool used_fallback).
    """
    # Filter indices where MeSH terms overlap with extracted concepts
    filtered_indices = [
        i for i, meta in enumerate(corpus["metadatas"])
        if _mesh_overlaps(meta, concepts)
    ]

    mesh_filtered_count = len(filtered_indices)
    used_fallback = False

    if mesh_filtered_count < min_filtered:
        # Fall back: use full corpus
        filtered_indices = list(range(len(corpus["ids"])))
        used_fallback = True

    # Embed the question
    query_vec = np.array(embeddings_model.embed_query(question), dtype=np.float32)

    # Select filtered embeddings and compute similarities
    filtered_embeddings = corpus["embeddings"][filtered_indices]
    similarities = _cosine_similarity(query_vec, filtered_embeddings)

    # Get top-k indices within the filtered set
    top_local_indices = np.argsort(similarities)[::-1][:k]
    top_global_indices = [filtered_indices[i] for i in top_local_indices]

    docs = [
        Document(
            page_content=corpus["documents"][idx],
            metadata=corpus["metadatas"][idx],
        )
        for idx in top_global_indices
    ]
    return docs, mesh_filtered_count, used_fallback

## Evaluation Functions

In [31]:
class GroqModel(DeepEvalBaseLLM):
    def __init__(self, model=None):
        self.model = model or get_groq_llm()

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        response = self.model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "Groq Model"


def build_test_cases(eval_df):
    return [
        LLMTestCase(
            input=row["question"],
            actual_output=row["generated_answer"],
            retrieval_context=row["retrieved_contexts"],
            expected_output=row["golden_answer"],
        )
        for _, row in eval_df.iterrows()
    ]


def _make_ragas_scores_df(all_scores, metric_name):
    """Return a minimal DataFrame with question_index and metric score."""
    return pd.DataFrame({
        "question_index": range(len(all_scores)),
        metric_name: all_scores,
    })


def evaluate_ragas(eval_df, metric, results_file=None):
    """Evaluate with a non-LLM RAGAS metric over each row of the eval DataFrame."""
    metric_name = type(metric).__name__
    all_scores = []

    for _, row in eval_df.iterrows():
        sample = SingleTurnSample(
            user_input=row["question"],
            retrieved_contexts=list(row["retrieved_contexts"]),
            reference_contexts=row["golden_contexts"],
            reference=row["golden_answer"],
            response=row["generated_answer"]
        )
        score = metric.single_turn_score(sample)
        all_scores.append(score)

    avg = sum(all_scores) / len(all_scores)
    print(f"\n=== {metric_name}: {avg:.4f} (avg over {len(all_scores)} samples) ===")

    scores_df = _make_ragas_scores_df(all_scores, metric_name)

    if results_file:
        scores_df.to_csv(results_file, index=False)
        print(f"Saved scores to {results_file}")

    return all_scores, avg, scores_df


def build_ragas_combined(eval_df, score_dfs, results_file=None):
    """Combine eval_df with per-metric score DataFrames into one summary CSV."""
    combined = eval_df.copy().reset_index(drop=True)
    combined.insert(0, "question_index", range(len(combined)))
    combined = combined.rename(columns={"generated_answer": "generated_response"})

    for scores_df in score_dfs:
        combined = combined.merge(scores_df, on="question_index", how="left")

    if results_file:
        combined.to_csv(results_file, index=False)
        print(f"Saved combined RAGAS results to {results_file}")

    return combined


def _run_deepeval_slice(test_case_slice, api_key, model, metric_cls, threshold, delay, key_idx, metric_kwargs=None):
    """Evaluate a contiguous slice of test cases using a single dedicated API key."""
    llm = ChatGroq(model=model, api_key=api_key)
    results = []

    for i, test_case in enumerate(test_case_slice):
        try:
            metric = metric_cls(threshold=threshold, model=GroqModel(model=llm),
                                **metric_kwargs)
            result = deepeval.evaluate([test_case], metrics=[metric],
                                       display_config=DisplayConfig(
                                           verbose_mode=False,
                                           show_indicator=False,
                                           print_results=False),
                                       async_config=AsyncConfig(run_async=False))
            results.extend(result.test_results)
        except Exception as e:
            print(f"[Key {key_idx}] Error on case {i + 1}/{len(test_case_slice)}: {e}")

        if i < len(test_case_slice) - 1:
            time.sleep(delay)

    print(f"[Key {key_idx}] Done — {len(results)}/{len(test_case_slice)} cases evaluated")
    return results


def evaluate_deepeval_parallel(test_cases, metric_cls, key_rotator, threshold=0.5, results_file=None, delay=None, rows_per_key=None, metric_kwargs=None):
    """Assign a contiguous slice of test cases to each API key and run all slices in parallel."""
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.DEEPEVAL_DELAY_SECONDS
    api_keys = key_rotator.api_keys
    metric_kwargs = {} if metric_kwargs is None else dict(metric_kwargs)

    if not test_cases:
        raise ValueError("test_cases is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(test_cases) > total_capacity:
        print(f"Warning: {len(test_cases)} cases exceed capacity. Truncating to {total_capacity}.")
        test_cases = test_cases[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(test_cases):
            break
        slices.append((key, i, test_cases[start: start + rows_per_key]))

    print(f"\n{len(test_cases)} cases split across {len(slices)} key(s) ({rows_per_key} cases/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: cases {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} cases)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_deepeval_slice,
                s, key, key_rotator.model,
                metric_cls, threshold, delay, i, metric_kwargs
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                ordered_results[idx] = []

    all_results = []
    res_df = None
    for key_results in ordered_results:
        all_results.extend(key_results)

    if all_results:
        scores = [r.metrics_data[0].score for r in all_results]
        metric_name = all_results[0].metrics_data[0].name
        average = sum(scores) / len(scores)
        print(f"\n=== {metric_name}: {average:.4f} (avg over {len(scores)} samples) ===")

        if results_file:
            rows = []
            for r in all_results:
                rows.append({
                    "question": r.input,
                    "generated_answer": r.actual_output,
                    "retrieved_contexts": r.retrieval_context,
                    "golden_answer": r.expected_output,
                    r.metrics_data[0].name: r.metrics_data[0].score,
                })
            res_df = pd.DataFrame(rows)
            res_df.to_csv(results_file, index=False)
            print(f"Saved to {results_file}")

    return all_results, res_df

## MeSH-Guided RAG Pipeline

The `_run_slice` function implements the 3-agent pipeline per row:
1. Agent 1 — Concept Extractor: one LLM call to identify medical concepts from the question.
2. Agent 2 — MeSH-Filtered Retriever: filters the pre-loaded corpus by MeSH overlap, then selects top-k by cosine similarity.
3. Agent 3 — Answer Generator: standard RAG generation using the filtered context.

In [14]:
def _run_slice(corpus, embeddings_model, slice_df, api_key, model, delay, key_idx, k=5):
    """Process a contiguous slice of the evaluation set using a single dedicated API key.

    Each row passes through the 3-agent MeSH-guided pipeline:
      Agent 1: concept extraction (1 LLM call)
      Agent 2: MeSH-filtered cosine retrieval (no LLM call)
      Agent 3: answer generation (1 LLM call)

    Args:
        corpus: Pre-loaded corpus dict from build_mesh_corpus.
        embeddings_model: HuggingFaceEmbeddings instance for query encoding.
        slice_df: Contiguous DataFrame slice assigned to this key.
        api_key: Groq API key string dedicated to this thread.
        model: LLM model name.
        delay: Seconds to sleep between rows.
        key_idx: Key index used only for log prefixes.
        k: Number of top chunks to retrieve per question.

    Returns:
        Copy of slice_df with new columns: extracted_concepts, retrieved_contexts,
        mesh_filtered_count, used_fallback, generated_answer, and timing/token columns.
    """
    llm = ChatGroq(model=model, api_key=api_key)
    concept_chain = build_concept_extraction_chain(llm)
    rag_chain = build_rag_chain(llm)

    result_df = slice_df.copy().reset_index(drop=True)
    extracted_concepts_list = [None] * len(slice_df)
    retrieved_contexts_list = [None] * len(slice_df)
    mesh_filtered_count_list = [None] * len(slice_df)
    used_fallback_list = [None] * len(slice_df)
    generated_answer_list = [None] * len(slice_df)
    total_time_list = [None] * len(slice_df)
    prompt_tokens_list = [None] * len(slice_df)
    completion_tokens_list = [None] * len(slice_df)
    total_tokens_list = [None] * len(slice_df)

    for row_idx, (_, row) in enumerate(slice_df.iterrows()):
        question = row["question"]
        try:
            time_start = time.perf_counter()

            # Agent 1: Extract medical concepts
            concept_result = concept_chain.invoke({"question": question})
            concepts = parse_concepts(concept_result)

            # Agent 2: MeSH-filtered cosine retrieval
            docs, mesh_count, fallback = mesh_filtered_retrieval(
                corpus, embeddings_model, question, concepts, k=k
            )

            # Agent 3: Answer generation
            gen_result = rag_chain.invoke({"context": docs, "question": question})
            total_time = time.perf_counter() - time_start

            # Token usage from the generation call (concept extraction tokens not captured separately)
            token_usage = gen_result.response_metadata.get("token_usage", {})

            extracted_concepts_list[row_idx] = concepts
            retrieved_contexts_list[row_idx] = [doc.page_content for doc in docs]
            mesh_filtered_count_list[row_idx] = mesh_count
            used_fallback_list[row_idx] = fallback
            generated_answer_list[row_idx] = gen_result.content
            total_time_list[row_idx] = total_time
            prompt_tokens_list[row_idx] = token_usage.get("prompt_tokens", 0)
            completion_tokens_list[row_idx] = token_usage.get("completion_tokens", 0)
            total_tokens_list[row_idx] = token_usage.get("total_tokens", 0)

        except Exception as e:
            print(f"[Key {key_idx}] Error on '{question[:50]}...': {e}")

        if row_idx < len(slice_df) - 1:
            time.sleep(delay)

    result_df["extracted_concepts"] = extracted_concepts_list
    result_df["retrieved_contexts"] = retrieved_contexts_list
    result_df["mesh_filtered_count"] = mesh_filtered_count_list
    result_df["used_fallback"] = used_fallback_list
    result_df["generated_answer"] = generated_answer_list
    result_df["total_time"] = total_time_list
    result_df["prompt_tokens"] = prompt_tokens_list
    result_df["completion_tokens"] = completion_tokens_list
    result_df["total_tokens"] = total_tokens_list

    completed = sum(1 for x in generated_answer_list if x is not None)
    print(f"[Key {key_idx}] Done — {completed}/{len(slice_df)} rows collected")
    return result_df


def run_mesh_rag_parallel(corpus, embeddings_model, df, key_rotator, rows_per_key=None, delay=None, k=5):
    """Assign a contiguous slice of rows to each API key and run all slices in parallel.

    Follows the same ThreadPoolExecutor strategy as all prior experiments.
    The pre-loaded corpus dict is shared across threads (read-only).

    Args:
        corpus: Pre-loaded corpus dict from build_mesh_corpus.
        embeddings_model: HuggingFaceEmbeddings instance for query encoding.
        df: DataFrame with at least question and golden_answer columns.
        key_rotator: GroqKeyRotator providing API keys and LLM model name.
        rows_per_key: Max rows assigned to each key (default config.PARALLEL_BUCKET_SIZE).
        delay: Seconds between rows within each key's slice (default config.PARALLEL_DELAY_SECONDS).
        k: Number of top chunks to retrieve per question.

    Returns:
        A copy of df with new columns added by _run_slice.
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or config.PARALLEL_DELAY_SECONDS
    api_keys = key_rotator.api_keys

    if len(df) == 0:
        raise ValueError("DataFrame is empty — nothing to evaluate.")

    total_capacity = len(api_keys) * rows_per_key
    if len(df) > total_capacity:
        print(f"Warning: {len(df)} rows exceed capacity ({total_capacity}). Truncating.")
        df = df.iloc[:total_capacity]

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(df):
            break
        slices.append((key, i, df.iloc[start: start + rows_per_key]))

    print(f"\n{len(df)} rows split across {len(slices)} key(s) ({rows_per_key} rows/key max):")
    for key, i, s in slices:
        print(f"  Key {i}: rows {i * rows_per_key}–{i * rows_per_key + len(s) - 1} ({len(s)} rows)")
    print()

    ordered_results = [None] * len(slices)

    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(
                _run_slice,
                corpus, embeddings_model, s, key, key_rotator.model, delay, i, k,
            ): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            _, _, s = slices[idx]
            try:
                ordered_results[idx] = future.result()
            except Exception as e:
                print(f"[Key {idx}] Thread failed: {e}")
                fallback_df = s.copy().reset_index(drop=True)
                for col in ["extracted_concepts", "retrieved_contexts", "generated_answer",
                             "mesh_filtered_count", "used_fallback"]:
                    fallback_df[col] = [None] * len(s)
                ordered_results[idx] = fallback_df

    final_df = pd.concat(ordered_results, ignore_index=True)
    completed = final_df["generated_answer"].notna().sum()
    print(f"\nCompleted {completed}/{len(df)} questions total")
    print(f"\nAverage Time Per Query: {final_df['total_time'].mean():.4f}s")
    print(f"\nAverage Total Tokens Per Query: {final_df['total_tokens'].mean():.1f}")
    fallback_count = final_df["used_fallback"].sum()
    print(f"\nFallback to full corpus: {fallback_count}/{len(df)} questions ({100*fallback_count/len(df):.1f}%)")
    print(f"\nAverage MeSH-filtered corpus size: {final_df['mesh_filtered_count'].mean():.1f} chunks")
    return final_df

---
## Setup

In [15]:
embedding_key = config.DEFAULT_EMBEDDING
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[embedding_key])
key_rotator = GroqKeyRotator()
llm = key_rotator.get_llm()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Embedding: {config.EMBEDDING_MODELS[embedding_key]}")
print(f"LLM: {key_rotator.model}")
print(f"Run timestamp: {timestamp}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initialized GroqKeyRotator with 10 API key(s)
Embedding: sentence-transformers/all-MiniLM-L6-v2
LLM: llama-3.3-70b-versatile
Run timestamp: 20260512_132445


## Prepare Evaluation Sample

Reads the pre-built golden dataset from `data/processed/golden_dataset_complete.csv`.

In [16]:
golden_df = pd.read_csv(config.DATA_PROCESSED_DIR / "golden_dataset_complete.csv")
print(f"Loaded {len(golden_df)} samples from golden_dataset_complete.csv")
golden_df['golden_contexts'] = golden_df['golden_contexts'].apply(literal_eval)
golden_df.info()

Loaded 200 samples from golden_dataset_complete.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   question_idx     200 non-null    int64 
 1   question         200 non-null    object
 2   golden_answer    200 non-null    object
 3   golden_contexts  200 non-null    object
 4   query_type       200 non-null    object
 5   pubids_needed    200 non-null    object
dtypes: int64(1), object(5)
memory usage: 9.5+ KB


## Load Vector Store and Build Mesh Corpus

In [17]:
vector_store = load_chroma(embeddings, db_name=f"{embedding_key}_pubmed_chromadb")

# Load all chunks once — shared across all parallel threads (read-only)
mesh_corpus = build_mesh_corpus(vector_store, embeddings)

Loading ChromaDB from /content/vectorstores/minilm_pubmed_chromadb
Loading corpus from ChromaDB (this is a one-time operation)...
Corpus loaded: 2849 chunks, docs = 2849


## Smoke Test: Single Question

In [18]:
smoke_result = run_mesh_rag_parallel(mesh_corpus, embeddings, golden_df.head(1), key_rotator)
smoke_result[["question", "extracted_concepts", "mesh_filtered_count", "used_fallback", "generated_answer"]]


1 rows split across 1 key(s) (20 rows/key max):
  Key 0: rows 0–0 (1 rows)

[Key 0] Done — 1/1 rows collected

Completed 1/1 questions total

Average Time Per Query: 1.2262s

Average Total Tokens Per Query: 1705.0

Fallback to full corpus: 0/1 questions (0.0%)

Average MeSH-filtered corpus size: 22.0 chunks


,question,extracted_concepts,mesh_filtered_count,used_fallback,generated_answer
0,Is there a relationship between rheumatoid art...,"[rheumatoid arthritis, periodontal disease, in...",22,False,"Yes, there is evidence to suggest a relationsh..."


## Run MeSH-Guided RAG on Full Evaluation Set

In [19]:
eval_dataset = run_mesh_rag_parallel(mesh_corpus, embeddings, golden_df, key_rotator)
eval_dataset.to_csv(
    str(config.RESULTS_EVALSETS_DIR / f"mesh_guided_rag_{embedding_key}_chroma_{timestamp}.csv"),
    index=False
)
print(f"Generated {len(eval_dataset)} answers")


200 rows split across 10 key(s) (20 rows/key max):
  Key 0: rows 0–19 (20 rows)
  Key 1: rows 20–39 (20 rows)
  Key 2: rows 40–59 (20 rows)
  Key 3: rows 60–79 (20 rows)
  Key 4: rows 80–99 (20 rows)
  Key 5: rows 100–119 (20 rows)
  Key 6: rows 120–139 (20 rows)
  Key 7: rows 140–159 (20 rows)
  Key 8: rows 160–179 (20 rows)
  Key 9: rows 180–199 (20 rows)

[Key 1] Done — 20/20 rows collected
[Key 4] Done — 20/20 rows collected
[Key 2] Done — 20/20 rows collected
[Key 6] Done — 20/20 rows collected
[Key 3] Done — 20/20 rows collected
[Key 5] Done — 20/20 rows collected
[Key 9] Done — 20/20 rows collected
[Key 7] Done — 20/20 rows collected
[Key 0] Done — 20/20 rows collected
[Key 8] Done — 20/20 rows collected

Completed 200/200 questions total

Average Time Per Query: 2.4677s

Average Total Tokens Per Query: 1691.2

Fallback to full corpus: 83/200 questions (41.5%)

Average MeSH-filtered corpus size: 48.9 chunks
Generated 200 answers


### RAGAS Evaluation

In [20]:
ragas_cr_scores, ragas_cr_avg, ragas_cr_df = evaluate_ragas(
    eval_dataset, NonLLMContextRecall(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"mesh_guided_rag_{embedding_key}_context_recall_{timestamp}.csv")
)


=== NonLLMContextRecall: 0.1888 (avg over 200 samples) ===
Saved scores to /content/results/ragas/mesh_guided_rag_minilm_context_recall_20260512_132445.csv


In [21]:
ragas_cp_scores, ragas_cp_avg, ragas_cp_df = evaluate_ragas(
    eval_dataset, NonLLMContextPrecisionWithReference(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"mesh_guided_rag_{embedding_key}_context_precision_{timestamp}.csv")
)


=== NonLLMContextPrecisionWithReference: 0.2754 (avg over 200 samples) ===
Saved scores to /content/results/ragas/mesh_guided_rag_minilm_context_precision_20260512_132445.csv


In [22]:
ragas_bleu_scores, ragas_bleu_avg, ragas_bleu_df = evaluate_ragas(
    eval_dataset, BleuScore(),
    results_file=str(config.RESULTS_RAGAS_DIR / f"mesh_guided_rag_{embedding_key}_bleu_{timestamp}.csv")
)


=== BleuScore: 0.1713 (avg over 200 samples) ===
Saved scores to /content/results/ragas/mesh_guided_rag_minilm_bleu_20260512_132445.csv


In [23]:
ragas_rouge_scores, ragas_rouge_avg, ragas_rouge_df = evaluate_ragas(
    eval_dataset, RougeScore(rouge_type="rougeL", mode="fmeasure"),
    results_file=str(config.RESULTS_RAGAS_DIR / f"mesh_guided_rag_{embedding_key}_rouge_{timestamp}.csv")
)


=== RougeScore: 0.3054 (avg over 200 samples) ===
Saved scores to /content/results/ragas/mesh_guided_rag_minilm_rouge_20260512_132445.csv


In [24]:
combined_ragas = build_ragas_combined(
    eval_dataset,
    [ragas_cr_df, ragas_cp_df, ragas_bleu_df, ragas_rouge_df],
    results_file=str(config.RESULTS_RAGAS_DIR / f"mesh_guided_rag_{embedding_key}_combined_{timestamp}.csv")
)

Saved combined RAGAS results to /content/results/ragas/mesh_guided_rag_minilm_combined_20260512_132445.csv


### DeepEval Evaluation

In [25]:
test_cases = build_test_cases(eval_dataset)
de_key_rotator = GroqKeyRotator(model="openai/gpt-oss-120b")

Initialized GroqKeyRotator with 10 API key(s)


In [ ]:
deepeval_cr, deepeval_cr_df = evaluate_deepeval_parallel(
    test_cases, ContextualRecallMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"mesh_guided_rag_{embedding_key}_ctx_recall_{timestamp}.csv")
)

In [ ]:
deepeval_cp, deepeval_cp_df = evaluate_deepeval_parallel(
    test_cases, ContextualPrecisionMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"mesh_guided_rag_{embedding_key}_ctx_precision_{timestamp}.csv")
)

In [35]:
deepeval_f, deepeval_f_df = evaluate_deepeval_parallel(
    test_cases, FaithfulnessMetric, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"mesh_guided_rag_{embedding_key}_faithfulness_{timestamp}.csv")
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



[Key 8] Done — 20/20 cases evaluated

=== Faithfulness: 0.9806 (avg over 195 samples) ===
Saved to /content/results/deepeval/mesh_guided_rag_minilm_faithfulness_20260512_132445.csv


In [33]:
# Defining Answer Correctness
evaluation_steps = [
    "Compare the generated answer with the reference answer in the context of the original biomedical question.",
    "Check whether the generated answer contains factually correct biomedical information and no contradictions to the reference answer.",
    "Verify that all clinically important facts needed to answer the question are present and no critical information is missing.",
    "Ignore wording differences, but penalize incorrect medical claims, unsupported conclusions, or misleading clinical interpretations."
]
deepeval_ac = evaluate_deepeval_parallel(
    test_cases, GEval, de_key_rotator,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / f"evidence_graded_rag_{embedding_key}_answer_correctness_{timestamp}.csv"),
    metric_kwargs={
        "name": "AnswerCorrectness",
        "evaluation_steps": evaluation_steps,
        "evaluation_params": [
            LLMTestCaseParams.INPUT,
            LLMTestCaseParams.ACTUAL_OUTPUT,
            LLMTestCaseParams.EXPECTED_OUTPUT,
        ]}
)


200 cases split across 10 key(s) (20 cases/key max):
  Key 0: cases 0–19 (20 cases)
  Key 1: cases 20–39 (20 cases)
  Key 2: cases 40–59 (20 cases)
  Key 3: cases 60–79 (20 cases)
  Key 4: cases 80–99 (20 cases)
  Key 5: cases 100–119 (20 cases)
  Key 6: cases 120–139 (20 cases)
  Key 7: cases 140–159 (20 cases)
  Key 8: cases 160–179 (20 cases)
  Key 9: cases 180–199 (20 cases)



[Key 7] Done — 20/20 cases evaluated

=== AnswerCorrectness [GEval]: 0.6700 (avg over 200 samples) ===
Saved to /content/results/deepeval/evidence_graded_rag_minilm_answer_correctness_20260512_132445.csv
